# FedSwarm — Phase 2.1 centralized ceiling (Colab GPU)

Trains the centralized performance ceiling every FL method in later phases gets measured against: `SimpleCNN` (primary config, 112px) across 5 seeds x {groupnorm, batchnorm}, plus an optional secondary `ResNet-18` @224 table.

**Before running:**
1. **Turn on GPU.** Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4). No account verification needed, unlike Kaggle.
2. **Have `archive (2).zip` (the Brain Tumor MRI dataset, ~164MB) ready to upload** -- the same file used to build the manifest locally. The cell below prompts an upload dialog; point it at that file from your Downloads folder.

This notebook does **not** need `flwr` -- that's only required once the Flower client/server harness exists (Phase 3+). Centralized training is plain PyTorch.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/content/ResearchPaper"

# Colab sessions can survive a cell re-run (e.g. retrying after an earlier cell
# failed) without wiping /content, so a plain `git clone` here fails with exit
# code 128 ("destination path already exists and is not an empty directory") on a
# rerun. Make this idempotent: pull if it's already a clone of this repo, re-clone
# if the directory exists but isn't (a partial/failed prior clone), else clone.
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
elif os.path.isdir(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

In [ ]:
%cd /content/ResearchPaper

# Colab's base image already has torch/torchvision/numpy/pandas/scikit-learn/
# scipy/matplotlib/pillow/tqdm. Only these are missing for centralized training
# (flwr/flwr-datasets/imagehash/kaggle are not imported by this training path).
!pip install -q omegaconf rich

# Install fedswarm itself (editable, no deps -- everything it needs is already
# satisfied above). Without this, `fedswarm` is only importable inside *this*
# notebook kernel process (via the sys.path hack a few cells down) -- every
# `!python -m fedswarm...` / `subprocess.run(["python", ...])` call below spawns a
# fresh interpreter that never sees that sys.path change and fails with
# `ModuleNotFoundError: No module named 'fedswarm'`. This is what silently broke
# the dataset download, cache build, and training steps on every prior Colab run.
!pip install -q -e . --no-deps

## Upload the dataset

Running this cell opens a file picker. Select `archive (2).zip` from your Downloads folder -- the same Brain Tumor MRI Dataset zip used locally to build `manifest.csv`. Upload takes a minute or two depending on your connection; the file is ~164MB.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
print(f"Uploaded: {zip_name} ({len(uploaded[zip_name]) / 1e6:.1f} MB)")

In [ ]:
import sys
sys.path.insert(0, "src")

import os
os.environ["FEDSWARM_DATA_ROOT"] = "/content/brain-tumor-mri"

!python -m fedswarm.data.download --zip "{zip_name}" --root "$FEDSWARM_DATA_ROOT"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type before continuing")

## Build the decoded-image cache

Uses the manifest already committed to the repo (`data/processed/manifest.csv`) -- the pseudo-patient-level split from Phase 1.3, with cross-split leakage verified zero. This step is I/O-bound (JPEG decode via PIL), not GPU-bound, and should take well under a minute for 7,200 images.

In [ ]:
!python -m fedswarm.data.cache --size 112

## Primary ceiling: SimpleCNN @112, 5 seeds x {groupnorm, batchnorm}

GroupNorm is the FL-relevant default; BatchNorm is run alongside as the A9 confound-control ablation (BatchNorm running statistics aggregate badly under non-IID federated data -- this comparison shows whether that's actually visible here, not just asserted).

In [ ]:
import subprocess

for norm in ["groupnorm", "batchnorm"]:
    for seed in [0, 1, 2, 3, 4]:
        print(f"\n{'='*20} norm={norm} seed={seed} {'='*20}")
        subprocess.run([
            "python", "scripts/run_experiment.py",
            "--config", "configs/base.yaml", "configs/model/simple_cnn.yaml", "configs/experiment/centralized.yaml",
            "--seed", str(seed),
            "--override", f"model.norm={norm}",
        ], check=True)

In [ ]:
!python scripts/summarize_centralized.py

## Optional: secondary ResNet-18 @224 table

"Does the ceiling hold with a larger, pretrained backbone." More expensive than the primary sweep (224px + 11.2M params) -- skip this cell entirely to save time/quota if you just need the primary ceiling.

In [ ]:
!python -m fedswarm.data.cache --size 224

for seed in [0, 1, 2, 3, 4]:
    print(f"\n{'='*20} resnet18 seed={seed} {'='*20}")
    subprocess.run([
        "python", "scripts/run_experiment.py",
        "--config", "configs/base.yaml", "configs/model/resnet18.yaml", "configs/experiment/centralized_resnet18.yaml",
        "--seed", str(seed),
    ], check=True)

!python scripts/summarize_centralized.py

## Getting results back

This zips `results/centralized/*.json` and triggers a browser download directly -- no output-tab hunting like Kaggle. Send the downloaded zip back for analysis.

In [ ]:
from google.colab import files

!cd results && zip -r /content/centralized_results.zip centralized/
files.download("/content/centralized_results.zip")